### (optional) Load dataset from HuggingFace

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` if your environment requires authentication

data = load_dataset("mozilla-foundation/common_voice_13_0", "zh-CN")

# Some modification
del data["invalidated"]
del data["other"]

data = data.remove_columns(["client_id", "up_votes", "down_votes", "age", "gender", "accent", "locale", "segment", "variant"])
# select_columns(["audio", "sentence"])
data

# Resample
# from datasets import Audio
# data = data.cast_column("audio", Audio(sampling_rate=16000))

## load Dataset         Step (0)

In [11]:
from datasets import load_from_disk

path_1 = "./prepared/CommonVoice13_V4"
path_2 = "../aishell-native/YOUR_AISHELL_DATASET_PATH"
data = load_from_disk(path_1)
# Display Audio
# from IPython.display import Audio
# item = data['train'][0]
# Audio(item['audio']['array'], rate=16000)x

print("The CommonVoice13-CN dataset sample.")
data["train"][0]

The CommonVoice13-CN dataset sample.


{'audio': {'path': 'common_voice_zh-CN_18551060.mp3',
  'array': array([-6.54836185e-11, -8.00355338e-11, -1.01863407e-10, ...,
         -2.31179220e-05, -2.89019226e-05, -1.07493543e-05]),
  'sampling_rate': 16000},
 'sentence': '巴顿是位于美国加利福尼亚州阿马多尔县的一个非建制地区。',
 'transcript_IPA': 'p a   t w ə n   ʂ ɨ   wei   y   m ei   k wɔ   tɕ ia   l i   f u   n i   ia   ʈʂ oʊ   a   m a   t wɔ   ɑɻ   ɕ iɛ n   t ɤ   i   k ɤ   f ei   tɕ iɛ n   ʈʂ ɨ   t ɤ   tɕʰ y',
 'tone_pinyin': '1 4 4 4 2 3 2 1 4 2 2 4 1 1 3 1 3 4 0 1 4 1 4 4 4 1',
 'transcript_IPAwithTone': 'p a1 t w ə4 n ʂ ɨ4 wei4 y2 m ei3 k wɔ2 tɕ ia1 l i4 f u2 n i2 ia4 ʈʂ oʊ1 a1 m a3 t wɔ1 ɑɻ3 ɕ iɛ4 n t ɤ i k ɤ f ei tɕ iɛ n ʈʂ ɨ t ɤ tɕʰ y'}

In [ ]:
import matplotlib.pyplot as plt

# Loop through the dataset to find a short sample (< 3 seconds)
for i, item in enumerate(data["train"]):
    audio_array = item["audio"]["array"]
    sampling_rate = item["audio"]["sampling_rate"]
    duration = len(audio_array) / sampling_rate

    if i == 9:
        continue

    if duration < 8 and duration > 5:
        print(f"Sample {i}: Duration = {duration:.2f} seconds")

        # Create time axis
        time_axis = [j / sampling_rate for j in range(len(audio_array))]

        # Plot signal
        plt.figure(figsize=(14, 4))
        plt.plot(time_axis, audio_array * 5)
        plt.title(f"Time-Domain Signal")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude")
        plt.ylim([-1, 1.2])
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        break  # Only show the first one

(data["train"][i])

from IPython.display import Audio

item = data["train"][i]
Audio(item["audio"]["array"], rate=16000)

#### Resize the dataset (optional Only Used if sampling_rate is not 16k)

In [ ]:
# ## Load a Processor to downsample the audio correctly
# from transformers import WhisperProcessor
# processor = WhisperProcessor.from_pretrained(
#     "openai/whisper-small", language="sinhalese", task="transcribe"
# )

# def prepare_dataset(example):
#     audio = example["audio"]

#     example = processor(
#         audio=audio["array"],
#         sampling_rate=audio["sampling_rate"],
#         text=example["sentence"],
#     )

#     # compute input length of audio sample in seconds
#     example["input_length"] = len(audio["array"]) / audio["sampling_rate"]

#     return example

# common_voice = common_voice.map(prepare_dataset)

# common_voice

## Chapter 1: DataPreProcessing -> Remove possible english,greek,other languages

In [ ]:
from datasets import load_from_disk
import unicodedata


def contains_non_chinese(sentence):
    """Check if the sentence contains any non-Chinese characters (e.g., English, Japanese, Greek, etc.)."""
    for char in sentence:
        # Check if the character is ASCII (English)
        if char.isascii():
            return True
        # Check if the character is Japanese (Hiragana, Katakana)
        if "\u3040" <= char <= "\u309f" or "\u30a0" <= char <= "\u30ff":
            return True
        # Check if the character is Greek (including μ)
        if "\u0370" <= char <= "\u03ff":
            return True
        # Check if the character is a mathematical symbol or other symbols
        if unicodedata.category(char).startswith("S"):
            return True
    return False


def filter_dataset(dataset):
    """Filter out entries with English or Japanese text in the 'sentence' column."""
    return dataset.filter(lambda example: not contains_non_chinese(example["sentence"]))


# Apply filtering to train, validation, and test datasets
data["train"] = filter_dataset(data["train"])
data["validation"] = filter_dataset(data["validation"])
data["test"] = filter_dataset(data["test"])

data

## 1) Add transcription into IPA format

In [ ]:
import re
from dragonmapper import hanzi


def remove_tones(ipa_string):
    # Define a pattern to match only tone markers (˥, ˧, ˩, etc.)
    pattern = r"[˥˧˩˦˨×]+"

    # Remove the tone markers using re.sub()
    ipa_string_no_tones = re.sub(pattern, "", ipa_string)

    return ipa_string_no_tones


def remove_punctuation(input_string):
    # Updated regex pattern to include a wide range of punctuation marks and spaces
    pattern = r"[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~·•《》「」『』【】…（）、；：！？——‘’“”，‧。“”·：、/ㄟＰ|・／－〉〈─□Λ]+"

    # Remove the unwanted characters
    cleaned_string = re.sub(pattern, "", input_string)

    return cleaned_string


# List of exceptions to keep together
exceptions = ["pʰ", "ts", "tsʰ", "tʰ", "ʈʂ", "ʈʂʰ", "tɕ", "tɕʰ", "kʰ", "ɑɻ", "ai", "ei", "ɑʊ", "oʊ", "ia", "iɛ", "wa", "wɔ", "ɥœ", "iɑʊ", "ioʊ", "wai", "wei"]


def convert_phoneme(input_string):
    try:
        # Remove punctuation
        input_string = remove_punctuation(input_string)

        # Convert to IPA
        ipa_result = hanzi.to_ipa(input_string, delimiter=" ", all_readings=False, container="[]")
        ipa_result = ipa_result.replace("j", "i").replace("ɪ", "i").replace("ń", "ən")

        # Remove tones and spaces from the IPA result
        ipa_result = remove_tones(ipa_result)
        ipa_result = " ".join(ipa_result.split())  # Removing all spaces

        symbols = list(ipa_result)
        result = []
        i = 0

        while i < len(symbols):

            # Check for syllable 'ɻ' and replace it with 'ɑɻ'
            if symbols[i] == "ɻ":
                result.append("ɑɻ")
                i += 1  # Move to the next character

            # Try matching 3-character exceptions first
            elif i < len(symbols) - 2 and (symbols[i] + symbols[i + 1] + symbols[i + 2]) in exceptions:
                result.append(symbols[i] + symbols[i + 1] + symbols[i + 2])
                i += 3  # Skip the next two characters since we've processed them as part of the exception
            # Try matching 2-character exceptions
            elif i < len(symbols) - 1 and (symbols[i] + symbols[i + 1]) in exceptions:
                result.append(symbols[i] + symbols[i + 1])
                i += 2  # Skip the next character since we've processed it as part of the exception
            else:
                # No match, so treat the current character as a separate symbol
                result.append(symbols[i])
                i += 1

        return " ".join(result)

    except ValueError as e:
        print(f"Error processing syllable: {input_string}")
        print(f"Error details: {e}")
        raise


# Define a function to add the 'transcript' column
def add_transcript_column(example):
    example["transcript_IPA"] = convert_phoneme(example["sentence"])
    return example


# Apply this function to add 'transcript' to the 'train', 'validation', and 'test' datasets
data["train"] = data["train"].map(add_transcript_column)
data["validation"] = data["validation"].map(add_transcript_column)
data["test"] = data["test"].map(add_transcript_column)

data

## 2) Add tone transcription to data in Pinyin

In [ ]:
import pinyin
import re


def remove_punctuation(input_string):
    # Updated regex pattern to include a wide range of punctuation marks and spaces
    pattern = r"[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~·•《》「」『』【】…（）、；：！？——‘’“”，‧。“”·：、/ㄟＰ|・／－〉〈─□Λ]+"

    # Remove the unwanted characters
    cleaned_string = re.sub(pattern, "", input_string)

    return cleaned_string


def convert_tone(a):

    a = remove_punctuation(a)

    a = pinyin.get(a, format="numerical")

    # Use a pattern to match digits only, which represent tones in Pinyin
    pattern = re.compile(r"[^\d]")
    # Replace all non-digit characters with an empty string
    a = pattern.sub("", a)
    a = " ".join(a)
    a = a.replace("5", "0")
    return a


def add_tone_column(example):
    example["tone_pinyin"] = convert_tone(example["sentence"])
    return example


data["train"] = data["train"].map(add_tone_column)
data["validation"] = data["validation"].map(add_tone_column)
data["test"] = data["test"].map(add_tone_column)

data

## 3) Add transcriptions which include IPA + tone (o1,o2,o3,o4)

In [ ]:
from pandas import read_csv

## initial settings
transcript_csv_file_path = "../../mandarin_p2a_model/data/IPAwithTone_p2attr_V2.csv"
attribute_mapping = read_csv(transcript_csv_file_path)


## Step 1) Read the vowel and consoant list in CSV dataframe
def read_Vowel_Consoant(csv_data):
    vowel_list, consoant_list = [], []
    for i in range(len(csv_data)):
        temp_list = attribute_mapping.iloc[i]
        if temp_list["vowel"] == 1:
            vowel_list.append(temp_list["Phoneme_ipaDragon"])
        else:
            consoant_list.append(temp_list["Phoneme_ipaDragon"])
    return vowel_list, consoant_list


vowel_list, consoant_list = read_Vowel_Consoant(attribute_mapping)


## Step 2) Start converting the datasets
def convert_IPAandTone(example):
    tone_list = example["tone_pinyin"].split()  ## Need adjust accordingly !!!!!!!
    temp_IPA_list = example["transcript_IPA"].split()  ## Need adjust accordingly !!!!!!!

    # Initialize tone index
    tone_index = 0
    # Iterate through `temp_IPA_list` and add tone to vowels
    for j, sb in enumerate(temp_IPA_list):
        if sb in vowel_list:
            if tone_index < len(tone_list):

                # Igonre the netural tone
                if tone_list[tone_index] == "0":
                    continue
                # Append tone to the vowel
                temp_IPA_list[j] = f"{sb}{tone_list[tone_index]}"
                tone_index += 1
            else:
                # In case there are more vowels than tones, handle as needed (e.g., skip or add default tone)
                temp_IPA_list[j] = f"{sb}"  # Keep the vowel as-is if no tone is available

    temp_IPA_list = " ".join(temp_IPA_list)
    return temp_IPA_list


def add_transcript_withToneIPA(batch):
    batch["transcript_IPAwithTone"] = convert_IPAandTone(batch)  # need changes
    return batch


data["train"] = data["train"].map(add_transcript_withToneIPA)
data["test"] = data["test"].map(add_transcript_withToneIPA)
data["validation"] = data["validation"].map(add_transcript_withToneIPA)

data["train"][210]

### Check Mapping unique occurence

In [ ]:
from collections import defaultdict


def count_unique_ipa_symbols(data):
    # Create a dictionary to store the count of each IPA symbol
    ipa_count = defaultdict(int)

    # Iterate over the 'transcript' column of the 'train' dataset
    i = 0
    for transcript in data["train"]["transcript_IPAwithTone"]:
        # Split the transcript into individual IPA symbols
        ipa_symbols = transcript.split()

        # Count each unique IPA symbol
        for symbol in ipa_symbols:
            if symbol == "μ":
                print(data["train"]["transcript_IPAwithTone"][i])
                print(data["train"]["sentence_speaker_said"][i])
            ipa_count[symbol] += 1
        i = i + 1

    # Convert the counts to a list of tuples (symbol, count)
    unique_ipa_list = list(ipa_count.items())

    return unique_ipa_list


# Example usage
unique_ipa_symbols = count_unique_ipa_symbols(data)

# Print the unique IPA symbols and their counts
for symbol, count in unique_ipa_symbols:
    print(f"Symbol: {symbol}, Count: {count}")

## 3) Resize the dataset

In [ ]:
from datasets import concatenate_datasets

# Load the original dataset
data = load_from_disk("./prepared/CommonVoice13_16K")

# Concatenate validation and test datasets to the train dataset
data["train"] = concatenate_datasets([data["train"], data["validation"], data["test"]])

# Optional: You can remove the validation and test splits if they are no longer needed
data.pop("validation")
data.pop("test")

data

### 3.2) Resize and shuffle the dataset

In [ ]:
from datasets import load_dataset, DatasetDict

# Step 1: Split the first 70% for training
train_test_split = data["train"].train_test_split(test_size=0.2, shuffle=False)

# Step 2: Split the remaining 30% into test and validation (14% and 15%)
test_valid_split = train_test_split["test"].train_test_split(test_size=0.5, shuffle=False)

# Step 3: Combine into a new DatasetDict with train, test, and validation sets
data = DatasetDict({"train": train_test_split["train"], "test": test_valid_split["train"], "validation": test_valid_split["test"]})  # 70% of the data  # 15% of the data  # 15% of the data
data

# train_size = 100
# test_size = 20
# validation_size = 20

# # Select subsets for each split
# data['train'] = data['train'].shuffle(42).select(range(train_size))
# data['validation'] = data['validation'].shuffle(23).select(range(validation_size))
# data['test'] = data['test'].shuffle(12).select(range(test_size))

# data

### 4) Save the new updated DATA

In [ ]:
new_path = "./prepared/CommonVoice13_V4"
data.save_to_disk(new_path)